# Sprints AI Career Orchestrator

## Final Demonstration Notebook

This notebook demonstrates the modular LangGraph project:

1. Curriculum Advisor
2. Python Assessor
3. Career Matching
4. Human-in-the-loop approval
5. SQLite persistence and automated tests

**Important:** Career openings are synthetic demonstration records.
Human reviewer identities in this notebook are simulated.


## 1. Environment Setup

Run this notebook from Google Colab after the project repository
has been cloned or copied to `/content/sprints-ai-orchestrator`.

Install the dependencies and configure `E2B_API_KEY` through
Colab Secrets or an environment variable before running the Assessor.


In [ ]:
from pathlib import Path
import os
import sys

candidates = [
    Path.cwd(),
    Path.cwd() / "sprints-ai-orchestrator",
    Path("/content/sprints-ai-orchestrator"),
    Path("/content/sprints_clean_test/sprints-ai-orchestrator"),
]

ROOT = next(
    (
        candidate.resolve()
        for candidate in candidates
        if (candidate / "src/orchestrator.py").is_file()
        and (candidate / "requirements.txt").is_file()
    ),
    None
)

if ROOT is None:
    raise FileNotFoundError(
        "Project not found. Open the notebook from the "
        "repository directory or clone the repository first."
    )

os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Project:", ROOT)
print("Python:", sys.version)


## 2. Verify the SQLite Dataset

In [ ]:
import sqlite3
from src.tools.database import DB_PATH

with sqlite3.connect(DB_PATH) as conn:
    learners = conn.execute(
        "SELECT COUNT(*) FROM learners"
    ).fetchone()[0]

    jobs = conn.execute(
        "SELECT COUNT(*) FROM job_openings"
    ).fetchone()[0]

print("Learners:", learners)
print("Synthetic jobs:", jobs)

assert learners >= 25
assert jobs >= 4


## 3. Curriculum Advisor

In [ ]:
import uuid
from src.orchestrator import final_orchestrator

advisor_result = final_orchestrator.invoke(
    {
        "task_type": "advisor",
        "learner_id": "SPR-1001",
        "question": "What is LangGraph used for?"
    },
    config={
        "configurable": {
            "thread_id": "advisor-" + str(uuid.uuid4())
        }
    }
)

print("Advisor answer:")
print(advisor_result.get("answer"))


## 4. Unsupported Curriculum Question

In [ ]:
unsupported_result = final_orchestrator.invoke(
    {
        "task_type": "advisor",
        "learner_id": "SPR-1001",
        "question": "What is the official Sprints refund policy?"
    },
    config={
        "configurable": {
            "thread_id": "unsupported-" + str(uuid.uuid4())
        }
    }
)

print(unsupported_result.get("answer"))


## 5. Python Assessor

This section runs the actual E2B sandbox and saves a new
assessment result in SQLite.

It requires a valid `E2B_API_KEY` and consumes sandbox credits.

**Run this cell only when you intend to execute a new assessment.**


In [ ]:
import os
import uuid

if not os.environ.get("E2B_API_KEY"):
    print("SKIPPED: Configure E2B_API_KEY before running.")
else:
    assessment_result = final_orchestrator.invoke(
        {
            "task_type": "assessor",
            "learner_id": "SPR-1001",
            "assignment_id": "PY-001",
            "code": (
                "def square_number(n):\n"
                "    return n * n\n"
            )
        },
        config={
            "configurable": {
                "thread_id": "assessor-" + str(uuid.uuid4())
            }
        }
    )

    print("Grade:", assessment_result.get("grade"))
    print("Feedback:", assessment_result.get("feedback"))
    print(
        "Saved result ID:",
        assessment_result.get("assessment_result_id")
    )


## 6. Career Matching and Human Approval

In [ ]:
import uuid
from langgraph.types import Command

career_config = {
    "configurable": {
        "thread_id": "career-" + str(uuid.uuid4())
    }
}

career_result = final_orchestrator.invoke(
    {
        "task_type": "career",
        "learner_id": "SPR-1001"
    },
    config=career_config
)

print("Matches:", career_result.get("matches"))
print("Approval status:", career_result.get("approval_status"))

snapshot = final_orchestrator.get_state(career_config)

print("Pending interrupts:", snapshot.interrupts)


## 7. Resume With a Simulated Reviewer Decision

This is a demonstration decision, not a real job application.
The decision is validated against the learner and selected job.


In [ ]:
if snapshot.interrupts:
    decision = {
        "learner_id": "SPR-1001",
        "job_id": "JOB-001",
        "approved": True,
        "reviewer_id": "DEMO-REVIEWER-001",
        "note": "Approved for demonstration."
    }

    approved_result = final_orchestrator.invoke(
        Command(resume=decision),
        config=career_config
    )

    print(
        "Final approval status:",
        approved_result.get("approval_status")
    )
else:
    print("No pending approval.")


## 8. Automated Regression Tests

In [ ]:
import subprocess
import sys

result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "tests/test_core.py",
        "-q"
    ],
    cwd=str(ROOT),
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr[-2000:])

assert result.returncode == 0


## 9. Limitations

- Checkpoints use in-memory storage and are not durable across runtime restarts.
- Career openings are synthetic, not verified live vacancies.
- The curriculum is demonstration content.
- Reviewer identities are simulated.
- The system requires additional security hardening for production.
- Reproducibility from a fresh runtime must be tested separately.
